In [1]:
import json
import datetime

## Exclude Period in Annotations

In [2]:
def period_checker(annotation_file):
    with open(annotation_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for ann in data['annotations']:
        for ent in ann[1]['entities']:
            start = ent[0]
            end = ent[1]
            ent_mention = ann[0][start:end]
            if ent_mention[-1] == '.':
                new_end = end - 1
                ent[1] = new_end
                print(f'Initial {end} ; Updated {new_end} ; "{ent_mention}"')

    return data

## Conversion to AnNER Format

In [3]:
def convert_to_anner(spacy_data, filename, annotator):
    STATUS = 'Candidate'
    COLOR_LIST = [
        'red-11', 'blue-11', 'light-green-11', 'yellow-11', 
        'purple-11', 'orange-11', 'teal-11', 'pink-11', 
        'brown-11', 'cyan-11', 'lime-11'
    ]
    
#     with open(input_file, 'r', encoding='utf-8') as file:
#         data = json.load(file)

    # Get timestamp
    timestamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

    # Build classes dynamically from input
    classes = []
    class_name_to_id = {}
    for idx, class_name in enumerate(spacy_data['classes'], start=1):
        color = COLOR_LIST[(idx - 1) % len(COLOR_LIST)]  # Rotate colors if needed
        classes.append({
            'id': idx,
            'name': class_name,
            'color': color
        })
        class_name_to_id[class_name] = class_name  # Just map to itself for annotation use

    # Build annotations
    annotations = []
    for text, ann in spacy_data['annotations']:
        entities = []
        for start, end, label in ann['entities']:
            entities.append([
                None,
                start,
                end,
                [
                    [
                        STATUS,
                        label,
                        timestamp,
                        annotator
                    ]
                ]
            ])
        annotations.append([None, text, {'entities': entities}])

    # Final structure
    anner_data = {
        'classes': classes,
        'annotations': annotations
    }
    
    # Write to output file
    with open(filename, 'w', encoding='utf-8') as file:
        json.dump(anner_data, file, indent=2)

    print(f'Conversion completed. Output saved to {filename}')


In [17]:
if __name__ == '__main__':
    convert_to_anner(
        spacy_data=period_checker('Robles-2015_spacy.json'),
        filename='Robles-2015_AnNER.json',
        annotator='gpt-4o'
    )

Conversion completed. Output saved to Robles-2015_AnNER.json
